# Paper 2 · Langkah 9: apakah dua temuan utama terulang?

Langkah 8 menunjukkan bahwa hasil dari satu kali latihan bisa tidak terulang. Notebook ini menguji ulang dua temuan yang masih berdiri di atas satu kali latihan.

**A. Matriks kompresi dengan model seed kedua.** Model dari langkah 8 diuji pada tiga versi gambar. Tidak ada latihan baru di bagian ini, kecuali lima model kondisi normal di bagian C.

**B. Uji lintas generator dengan tiga seed tambahan**, untuk ketiga kondisi dan kedua arah (18 model).

**C. Kondisi normal seed kedua** (5 model), supaya matriks seed kedua lengkap tiga baris. Dikerjakan paling akhir.

**Aturan baca yang dikunci sebelum hasil:**
- *Perpindahan jalan pintas* dianggap terulang kalau, pada model satu sisi seed kedua, gambar Stable Diffusion versi simetris dikenali minimal 10 poin lebih rendah daripada versi satu sisi, **dan** karya manusia versi normal dikenali minimal 10 poin lebih rendah daripada versi satu sisi. (Seed pertama: 22,6 dan 16,8 poin.)
- *Perbaikan lintas generator* dianggap terulang kalau, di tiap arah, rata-rata empat seed untuk kondisi simetris lebih tinggi daripada kondisi satu sisi dan kondisi normal, **dan** simetris mengungguli satu sisi pada minimal 3 dari 4 seed.
- Kalau salah satu gagal, klaim yang bersangkutan di naskah dilemahkan.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**. Perkiraan sekitar 50 menit. Hasil tiap bagian langsung disimpan ke Drive; kalau sesi terputus, klik Run all lagi.

## 1. Persiapan dan cek GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, random, shutil, zipfile, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.transforms.functional import gaussian_blur
import cv2, scipy, sklearn
from scipy import stats, ndimage
from PIL import Image
import PIL

# Pengaturan (sama dengan langkah 3 sampai 6; jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
SEED2 = 1042                      # seed kedua (langkah 8)
EXTRA_SEEDS = [1052, 2052, 3052]  # seed tambahan untuk uji lintas generator (seed pertama: 52)
INPUT = 224
PATCH, STRIDE = 32, 16
TOP = 0.20
N_RANDOM = 10
BLUR_KERNEL, BLUR_SIGMA = 21, 7.0
HUE_BINS, GLCM_LEVELS = 36, 32
CANNY_LOW, CANNY_HIGH = 100, 200
CHROMA_MIN = 0.10
MAX_EPOCHS, PATIENCE, BATCH, LR, LR_MIN, PRETRAINED = 30, 5, 32, 1e-4, 1e-6, True

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
RUNS = RESULTS / 'detektor_runs'
MODELS = RESULTS / 'models'
MAPS3 = RESULTS / 'peta_simetris'
ELEM3 = RESULTS / 'elemen_simetris'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [RUNS, MODELS, MAPS3, ELEM3, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'
USE_AMP = device == 'cuda'

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}\nsklearn: {sklearn.__version__}\n'
       f'opencv: {cv2.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'seed lintas generator tambahan: {EXTRA_SEEDS}; matriks kompresi dengan model seed {SEED2}\n')
(NOTES / 'lingkungan_langkah9.txt').write_text(env)
print(env)

## 2. Data: tiga versi kompresi

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)
assert (RESULTS / 'rq1_matriks_kompresi.csv').exists(), 'Jalankan notebook langkah 7 dulu.'
assert all((MODELS / f'{c}_s2_fold{k}.pt').exists() for c in ('simetris', 'riwayat_disamakan') for k in range(5)), 'Jalankan notebook langkah 8 dulu.'

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)

def enc(im, q, sub):
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q, subsampling=sub, optimize=False); return buf.getvalue()
def dec(b):
    im = Image.open(io.BytesIO(b)); im.load(); return im.convert('RGB')
def square256(im):
    im = im.convert('RGB'); w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    return im

def make_version(row, version):
    with Image.open(row.raw_path) as im:
        im.load(); im = im.convert('RGB')
    if version == 'riwayat_disamakan' and row.source == 'human':
        im = dec(enc(im, 75, 2))
    im = square256(im)
    if version == 'simetris':
        im = dec(enc(im, 75, 2))
    b = enc(im, 90, 0)
    out = dec(b)
    if out.size != (INPUT, INPUT):
        out = out.resize((INPUT, INPUT), Image.BICUBIC)
    return np.asarray(out, dtype=np.uint8), len(b)

VERSIONS = ['normal', 'riwayat_disamakan', 'simetris']
XH, XT, SIZE = {}, {}, {}
for v in VERSIONS:
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(lambda r: make_version(r, v), manifest.itertuples()))
    XH[v] = np.stack([a for a, _ in res]); SIZE[v] = np.array([s for _, s in res]) / 1024
    XT[v] = torch.from_numpy(XH[v]).permute(0, 3, 1, 2).contiguous()
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
src_all, folds, y_all = manifest['source'].values, manifest['fold'].values, manifest['y'].values
strata_all = (manifest['source'] + '|' + manifest['style']).values
print(f'Data siap ({time.time() - t0:.0f} detik):', {v: XH[v].shape for v in VERSIONS})

## 3. Fungsi model dan latihan (sama dengan langkah 7)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
NPIX = INPUT * INPUT
FRACS = torch.arange(1, 11, device=device) * 0.05

def norm_t(xb):
    return ((xb.to(device).float().div(255) - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def build_model(pretrained=False):
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device).to(memory_format=torch.channels_last)

def load_model(name):
    m = build_model(False)
    m.load_state_dict(torch.load(MODELS / f'{name}.pt', map_location=device))
    return m.eval()

@torch.no_grad()
def probs(model, xn, bs=256):
    out = []
    for i in range(0, len(xn), bs):
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(xn[i:i + bs])
        out.append(torch.softmax(logits.float(), 1))
    return torch.cat(out)

@torch.no_grad()
def predict_ai(model, X, idx):
    return torch.cat([probs(model, norm_t(X[idx[i:i + 128]])) for i in range(0, len(idx), 128)]).cpu().numpy()

@torch.no_grad()
def logits32(model, xn, bs=256):
    return torch.cat([model(xn[i:i + bs]).float() for i in range(0, len(xn), bs)])

# --- latihan (sama dengan langkah 3 dan 4) ---
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def prep_train(xb):
    xb = xb.to(device, non_blocking=True).float().div_(255)
    flip = torch.rand(xb.size(0), device=device) < 0.5
    xb[flip] = xb[flip].flip(-1)
    return ((xb - MEAN) / STD).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def val_loss(model, X, idx):
    model.eval(); tot = 0.0
    for i in range(0, len(idx), 128):
        b = idx[i:i + 128]
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(norm_t(X[b]))
        tot += F.cross_entropy(logits.float(), torch.as_tensor(y_all[b], device=device), reduction='sum').item()
    return tot / len(idx)

def train_model(X, tr_idx, va_idx, seed):
    set_seed(seed)
    model = build_model(PRETRAINED)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    rng = np.random.default_rng(seed)
    best, best_state, bad, hist = float('inf'), None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); perm = rng.permutation(tr_idx)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=USE_AMP):
                loss = F.cross_entropy(model(prep_train(X[b])), torch.as_tensor(y_all[b], device=device))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v = val_loss(model, X, va_idx); hist.append(round(v, 4))
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model.eval(), hist

def inner_split(tr_idx, seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    try:
        a, b = next(sss.split(tr_idx, strata_all[tr_idx]))
    except ValueError:
        a, b = next(sss.split(tr_idx, src_all[tr_idx]))
    return tr_idx[a], tr_idx[b]

def source_rates(p_ai, idx):
    pred = (p_ai >= 0.5).astype(int); s = src_all[idx]; y = y_all[idx]; out = {'akurasi': float((pred == y).mean())}
    if len(np.unique(y)) == 2: out['auc'] = float(roc_auc_score(y, p_ai))
    for k, name in (('human', 'benar_manusia'), ('latent_diffusion', 'benar_LD'), ('stable_diffusion', 'benar_SD')):
        if (s == k).any(): out[name] = float((pred[s == k] == y[s == k]).mean())
    return out

# --- peta occlusion skala logit dan uji deletion (sama dengan langkah 6) ---
pos = list(range(0, INPUT - PATCH + 1, STRIDE)); N1 = len(pos)
M1 = torch.zeros(N1 * N1, 1, INPUT, INPUT, dtype=torch.bool, device=device); k = 0
for i in pos:
    for j in pos:
        M1[k, 0, i:i + PATCH, j:j + PATCH] = True; k += 1

@torch.no_grad()
def occlusion_logit(model, xn, pred):
    out, m0s = [], []
    for i in range(len(xn)):
        x = xn[i:i + 1]; c = int(pred[i]); mu = x.mean((2, 3), keepdim=True)
        z0 = logits32(model, x)[0]; m0 = z0[c] - z0[1 - c]
        z = logits32(model, torch.where(M1, mu, x))
        out.append((m0 - (z[:, c] - z[:, 1 - c])).reshape(N1, N1)); m0s.append(m0)
    return torch.stack(out), torch.stack(m0s)

def patch_to_cells(d):
    kk, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C, device=d.device); cnt = torch.zeros(C, C, device=d.device)
    for a in range(kk):
        for b in range(kk):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_desc(maps, gen):
    flat = maps.reshape(len(maps), -1)
    flat = flat + 1e-7 * flat.abs().amax(1, keepdim=True).clamp(min=1e-12) * torch.rand(flat.shape, device=flat.device, generator=gen)
    order = flat.argsort(1, descending=True)
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(order.shape[1], device=flat.device).expand_as(order))
    return rank

yy, xx = torch.meshgrid(torch.arange(INPUT, device=device), torch.arange(INPUT, device=device), indexing='ij')
c0 = (INPUT - 1) / 2
CENTRE = 1 - torch.sqrt((yy - c0) ** 2 + (xx - c0) ** 2) / (c0 * 2 ** 0.5)

@torch.no_grad()
def deletion(model, x, maps, c, p0, gen):
    rank = rank_desc(maps, gen)
    thr = (FRACS * NPIX).long()
    mask = (rank[:, None, :] < thr[None, :, None]).reshape(-1, 1, INPUT, INPUT)
    out = {}
    for name, rep in (('blur', gaussian_blur(x, [BLUR_KERNEL, BLUR_KERNEL], [BLUR_SIGMA, BLUR_SIGMA])), ('mean', x.mean((2, 3), keepdim=True))):
        p = probs(model, torch.where(mask, rep, x))[:, c].reshape(len(maps), -1)
        out[name] = (0.5 * p0 + p[:, :-1].sum(1) + 0.5 * p[:, -1]) / len(FRACS)
    return out
print('Fungsi siap. Posisi kotak occlusion:', N1, 'x', N1)

## 4. Bagian A: matriks kompresi dengan model seed kedua (cepat)

In [ ]:
SRCMAP = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
NAMA = {'normal': 'normal', 'riwayat_disamakan': 'satu sisi', 'simetris': 'simetris'}

def matrix_rows(fam):
    rows = []
    for k in range(5):
        model = load_model(f'{fam}_s2_fold{k}'); te = np.where(folds == k)[0]
        for v in VERSIONS:
            rows.append({'model': fam, 'gambar': v, 'fold': k, **source_rates(predict_ai(model, XT[v], te)[:, 1], te)})
        del model; torch.cuda.empty_cache()
    return rows

mat2_file = RESULTS / 'rq1_matriks_kompresi_seed2.csv'
MAT2 = pd.read_csv(mat2_file) if mat2_file.exists() else pd.DataFrame()
for fam in ('riwayat_disamakan', 'simetris'):
    if len(MAT2) and (MAT2.model == fam).any():
        print(f'Matriks seed kedua, model {NAMA[fam]}: sudah selesai, dilewati.'); continue
    MAT2 = pd.concat([MAT2, pd.DataFrame(matrix_rows(fam))], ignore_index=True); MAT2.to_csv(mat2_file, index=False)
    print(f'Matriks seed kedua, model {NAMA[fam]}: selesai.')

## 5. Bagian B: uji lintas generator dengan tiga seed tambahan (bagian terlama)

In [ ]:
SHORT = {'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
human_tr = np.where((src_all == 'human') & (folds != 4))[0]; human_te = np.where((src_all == 'human') & (folds == 4))[0]
for seed in EXTRA_SEEDS:
    for cond in ('simetris', 'riwayat_disamakan', 'normal'):
        for a, b in [('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')]:
            name = f'lintas9_{cond}_latih{SHORT[a]}_uji{SHORT[b]}_seed{seed}'
            if (RUNS / f'{name}.json').exists():
                print(f'{name}: sudah selesai, dilewati.'); continue
            t0 = time.time(); X = XT[cond]
            tr_idx = np.concatenate([human_tr, np.where(src_all == a)[0]]); te_idx = np.concatenate([human_te, np.where(src_all == b)[0]])
            tr, va = inner_split(tr_idx, seed)
            model, hist = train_model(X, tr, va, seed)
            p = predict_ai(model, X, te_idx)[:, 1]
            rec = {'nama': name, 'kondisi': cond, 'arah': f'latih {SHORT[a]}, uji {SHORT[b]}', 'seed': seed, 'epoch': len(hist), 'detik': round(time.time() - t0),
                   'ai_dikenali': float((p[y_all[te_idx] == 1] >= 0.5).mean()), **source_rates(p, te_idx)}
            (RUNS / f'{name}.json').write_text(json.dumps(rec))
            print(f"{name}: AI dikenali {100 * rec['ai_dikenali']:.1f}%, AUC {rec['auc']:.3f}, manusia benar {rec['benar_manusia']:.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
            del model; torch.cuda.empty_cache()

## 6. Bagian C: kondisi normal seed kedua, lalu baris ketiga matriks

In [ ]:
for k in range(5):
    name = f'normal_s2_fold{k}'
    if (MODELS / f'{name}.pt').exists() and (RUNS / f'{name}.json').exists():
        print(f'{name}: sudah selesai, dilewati.'); continue
    t0 = time.time(); X = XT['normal']; seed = SEED2 + k
    tr_idx, te_idx = np.where(folds != k)[0], np.where(folds == k)[0]
    tr, va = inner_split(tr_idx, seed)
    model, hist = train_model(X, tr, va, seed)
    p = predict_ai(model, X, te_idx)[:, 1]
    rec = {'nama': name, 'kondisi': 'normal', 'seed': seed, 'epoch': len(hist), 'detik': round(time.time() - t0), **source_rates(p, te_idx)}
    torch.save(model.state_dict(), MODELS / f'{name}.pt'); (RUNS / f'{name}.json').write_text(json.dumps(rec))
    print(f"{name}: akurasi {rec['akurasi']:.3f}, AUC {rec['auc']:.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
    del model; torch.cuda.empty_cache()

MAT2 = pd.read_csv(mat2_file)
if (MAT2.model == 'normal').any():
    print('Matriks seed kedua, model normal: sudah selesai, dilewati.')
else:
    MAT2 = pd.concat([MAT2, pd.DataFrame(matrix_rows('normal'))], ignore_index=True); MAT2.to_csv(mat2_file, index=False)
    print('Matriks seed kedua, model normal: selesai.')

## 7. Ringkasan dan pemeriksaan aturan baca

In [ ]:
COLS = ['benar_manusia', 'benar_LD', 'benar_SD', 'akurasi']
def show_matrix(M, title):
    fams = [f for f in VERSIONS if (M.model == f).any()]
    t = M.groupby(['model', 'gambar'])[COLS].mean().reindex(pd.MultiIndex.from_product([fams, VERSIONS], names=['model', 'gambar']))
    t.index = pd.MultiIndex.from_tuples([(NAMA[a], NAMA[b]) for a, b in t.index], names=['model dilatih pada', 'gambar diuji'])
    print(title); print(t.round(3).to_string())
    return M.groupby(['model', 'gambar'])[COLS].mean()

MAT1 = pd.read_csv(RESULTS / 'rq1_matriks_kompresi.csv'); MAT2 = pd.read_csv(RESULTS / 'rq1_matriks_kompresi_seed2.csv')
m1 = show_matrix(MAT1, 'M1. MATRIKS KOMPRESI, seed pertama (langkah 7)')
print(); m2 = show_matrix(MAT2, 'M2. MATRIKS KOMPRESI, seed kedua')

def drops(m):
    o = 'riwayat_disamakan'
    return 100 * (m.loc[(o, o), 'benar_SD'] - m.loc[(o, 'simetris'), 'benar_SD']), 100 * (m.loc[(o, o), 'benar_manusia'] - m.loc[(o, 'normal'), 'benar_manusia'])
d1, d2 = drops(m1), drops(m2)
print('\nM3. ATURAN BACA, perpindahan jalan pintas (model satu sisi; penurunan dalam poin persentase)')
print(pd.DataFrame({'seed pertama': d1, 'seed kedua': d2}, index=['SD: versi satu sisi ke versi simetris', 'manusia: versi satu sisi ke versi normal']).round(1).to_string())
print(f"   Seed kedua: {'TERULANG' if min(d2) >= 10 else 'TIDAK TERULANG'} (syarat: kedua penurunan minimal 10 poin)")

# ---- lintas generator ----
REF = {('normal', 'latih LD, uji SD'): (0.001, 0.498), ('normal', 'latih SD, uji LD'): (0.013, 0.486),
       ('riwayat_disamakan', 'latih LD, uji SD'): (0.049, 0.751), ('riwayat_disamakan', 'latih SD, uji LD'): (0.035, 0.630),
       ('simetris', 'latih LD, uji SD'): (0.221, 0.824), ('simetris', 'latih SD, uji LD'): (0.153, 0.792)}   # seed pertama, dari langkah 3 dan 7
rows = [{'kondisi': c, 'arah': a, 'seed': 52, 'ai_dikenali': v[0], 'auc': v[1]} for (c, a), v in REF.items()]
for f in sorted(RUNS.glob('lintas9_*.json')):
    r = json.loads(f.read_text()); rows.append({k: r[k] for k in ('kondisi', 'arah', 'seed', 'ai_dikenali', 'auc', 'benar_manusia')})
L = pd.DataFrame(rows); L.to_csv(RESULTS / 'rq1_lintas_generator_per_seed.csv', index=False)
print(f'\nL1. LINTAS GENERATOR: persen gambar AI dari generator tak dikenal yang dikenali, per seed ({L.seed.nunique()} seed)')
for arah, g in L.groupby('arah'):
    t = (100 * g.pivot(index='seed', columns='kondisi', values='ai_dikenali')).reindex(columns=['normal', 'riwayat_disamakan', 'simetris'])
    t.columns = [NAMA[c] for c in t.columns]
    t.loc['rata-rata'] = t.mean(); t.loc['terendah'] = t.iloc[:-1].min(); t.loc['tertinggi'] = t.iloc[:-2].max()
    print(f'\n   {arah}'); print(t.round(1).to_string())
print('\nL2. LINTAS GENERATOR: AUC per seed')
for arah, g in L.groupby('arah'):
    t = g.pivot(index='seed', columns='kondisi', values='auc').reindex(columns=['normal', 'riwayat_disamakan', 'simetris']); t.columns = [NAMA[c] for c in t.columns]
    t.loc['rata-rata'] = t.mean(); print(f'\n   {arah}'); print(t.round(3).to_string())
if 'benar_manusia' in L:
    print(f"\n   Karya manusia benar pada uji lintas generator (seed tambahan): {100 * L.benar_manusia.min():.1f}% sampai {100 * L.benar_manusia.max():.1f}%")

print('\nL3. ATURAN BACA, perbaikan lintas generator')
ok_all = True
for arah, g in L.groupby('arah'):
    p = g.pivot(index='seed', columns='kondisi', values='ai_dikenali').dropna()
    mean_ok = p['simetris'].mean() > p['riwayat_disamakan'].mean() and p['simetris'].mean() > p['normal'].mean()
    wins = int((p['simetris'] > p['riwayat_disamakan']).sum())
    ok = mean_ok and wins >= 3 and len(p) >= 4; ok_all &= ok
    print(f"   {arah}: rata-rata simetris {100 * p['simetris'].mean():.1f}%, satu sisi {100 * p['riwayat_disamakan'].mean():.1f}%, normal {100 * p['normal'].mean():.1f}%; simetris unggul atas satu sisi di {wins} dari {len(p)} seed -> {'lolos' if ok else 'gagal'}")
print(f"   Kesimpulan: {'TERULANG' if ok_all else 'TIDAK TERULANG'}")
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tabel M1 sampai M3 dan L1 sampai L3 muncul, semua beres. Tekan **Cmd+S**, lalu bilang "sudah" ke Claude dengan link Colab-nya.